# OPRB benchmark: Qwen/Qwen2.5-VL-3B-Instruct

Compact, reproducible test-set evaluation.

In [ ]:
%pip -q install "transformers>=4.53,<5" accelerate bitsandbytes qwen-vl-utils "huggingface_hub>=0.30" "jiwer>=3" pandas Pillow tqdm

In [ ]:
MODEL_TAG = 'Qwen2_5_VL_3B'
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

import json, random, re, time, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm.auto import tqdm

REPO_ID = "kpurkayastha/OPRB"
DATA_ROOT = Path("/content/oprb_test")
OUTPUT_DIR = Path("/content/oprb_results") / MODEL_TAG
CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = None  # Evaluate all 9,000 pages.
RANDOM_SEED = 42
NORMALIZE_CASE = True
SAVE_EVERY = 25
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _extract(archive_path, destination, suffix):
    import shutil, tarfile
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive:
            if member.isfile() and member.name.lower().endswith(suffix):
                target = destination / Path(member.name).name
                if not target.exists():
                    with archive.extractfile(member) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst)

def download_data():
    for category in CATEGORIES:
        for kind, suffix, expected in [("img", ".jpg", 6000), ("gt", ".txt", 6000)]:
            if category == "nontext" and kind in ("img", "gt"): expected = 3000
            folder = DATA_ROOT / category / kind
            if len(list(folder.glob(f"*{suffix}"))) < expected:
                archive = hf_hub_download(REPO_ID, f"test/{category}/{kind}.tar.gz", repo_type="dataset")
                _extract(archive, folder, suffix)

def collect_pairs():
    pairs = []
    for category in CATEGORIES:
        image_dir, gt_dir = DATA_ROOT / category / "img", DATA_ROOT / category / "gt"
        names = sorted({p.stem for p in image_dir.glob("*.jpg")} & {p.stem for p in gt_dir.glob("*.txt")})
        if MAX_SAMPLES_PER_CATEGORY is not None:
            names = sorted(random.Random(RANDOM_SEED).sample(names, min(len(names), MAX_SAMPLES_PER_CATEGORY)))
        for stem in names:
            parts = stem.split("__", 2)
            pairs.append({"category": category, "sample_id": stem, "image_path": str(image_dir/(stem+".jpg")), "gt_path": str(gt_dir/(stem+".txt")), "occlusion_type": parts[0], "occlusion_density": parts[1] if len(parts)>1 else "unknown"})
    if not pairs: raise RuntimeError("No paired OPRB files found")
    return pairs

def gt_text(path):
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        return " ".join(line.rstrip("\r\n").split("\t")[0].strip() for line in f if "\t" in line and line.split("\t",1)[0].strip())

def norm(s):
    s = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s)).strip()
    return s.casefold() if NORMALIZE_CASE else s

def metrics(ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    if not ref: raise ValueError("empty ground truth")
    a, b = Counter(ref.split()), Counter(hyp.split())
    matched = sum((a & b).values())
    p = matched / sum(b.values()) if b else 0.0
    r = matched / sum(a.values()) if a else 0.0
    return {"cer": float(cer(ref,hyp)), "wer": float(wer(ref,hyp)), "token_precision": p, "token_recall": r, "token_f1": 2*p*r/(p+r) if p+r else 0.0}

download_data()
pairs = collect_pairs()


In [ ]:
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
MODEL_ID="Qwen/Qwen2.5-VL-3B-Instruct"
processor=AutoProcessor.from_pretrained(MODEL_ID)
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto").eval()

In [ ]:
def transcribe_page(path):
    start=time.perf_counter(); image=Image.open(path).convert("RGB"); messages=[{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":"Transcribe all visible text in reading order. Output only the transcription."}]}]
    inputs=processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode(): ids=model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    text=processor.decode(ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text.strip(), time.perf_counter()-start, int(ids.shape[-1]), ids.shape[-1] >= inputs["input_ids"].shape[-1]+1024


In [ ]:

RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"
old = pd.read_csv(RESULTS_CSV).to_dict("records") if RESULTS_CSV.exists() else []
rows = [r for r in old if r.get("sample_id") in {p["sample_id"] for p in pairs}]
finished = {r["sample_id"] for r in rows if r.get("status") == "ok"}
for i, sample in enumerate(tqdm(pairs), 1):
    if sample["sample_id"] in finished: continue
    row = {**sample, "model_id": MODEL_ID}
    try:
        prediction, latency, generated, truncated = transcribe_page(sample["image_path"])
        row.update({"status":"ok", "prediction":prediction, "ground_truth":gt_text(sample["gt_path"]), "latency_sec":latency, "generated_tokens":generated, "possibly_truncated":truncated})
        row.update(metrics(row["ground_truth"], prediction))
    except Exception as exc:
        row.update({"status":"error", "error":repr(exc)})
    rows.append(row)
    if i % SAVE_EVERY == 0: pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
results = pd.DataFrame(rows)
ok = results[results.status == "ok"]
display(ok[["category","cer","wer","token_f1","latency_sec"]].describe(include="all") if len(ok) else results)
